# Week 11: Models You Can Read: Decision Trees

**Grade band:** 6 to 8  |  **Duration:** 60 minutes  |  **Platform:** JupyterLite (browser, no account) or Google Colab

**How to use this notebook:** run each cell from top to bottom with Shift + Enter. Read the text, run the code, then complete the challenge cells marked **SOLUTION**. Save your work at the end of the session (File > Download) so it can be uploaded to your portfolio.

> **Teacher copy.** This notebook contains completed answers for every tier. Do not distribute to students. Use it to check work against the validation checklist.

## Hook: Twenty questions

Think of an animal. Your partner gets yes or no questions only. Every good question splits the possibilities in half. A **decision tree** is a computer playing twenty questions with your data, and today you get to read every question it chose.

In [ ]:
# Setup: run this cell first.
import pandas as pd
import matplotlib.pyplot as plt

# If a data file is not found next to this notebook (for example on Google Colab),
# it is loaded from the Wize data folder online instead. Replace this URL after publishing.
DATA_URL = "https://raw.githubusercontent.com/wizeacademy/ml-ai-6-8/main/notebooks/data/"

def load(name):
    """Load a Wize dataset by file name, from the local data folder or from the web."""
    try:
        return pd.read_csv("data/" + name)
    except Exception:
        return pd.read_csv(DATA_URL + name)

print("Setup complete. pandas and matplotlib are ready.")

## Teach 1: Train a tree and read it

Same recipe as K nearest neighbors, different model. The difference: a tree can **explain itself**. `plot_tree` draws every question.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split

penguins = load("penguins.csv").dropna()
features = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
X = penguins[features]
y = penguins["species"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

tree = DecisionTreeClassifier(max_depth=2, random_state=42)
tree.fit(X_train, y_train)
print("Accuracy with depth 2:", round(tree.score(X_test, y_test) * 100, 1), "%")

plt.figure(figsize=(12, 5))
plot_tree(tree, feature_names=features, class_names=tree.classes_, filled=True, fontsize=9)
plt.show()

## Teach 2: Deeper is not always better

`max_depth` is how many questions the tree may ask in a row. A very deep tree memorizes the training penguins, including their quirks, and then does worse on new ones. That is **overfitting**. Watch the two accuracies as depth grows.

**Concept checkpoint:** predict which depth will have the highest *training* accuracy. Then predict which will have the highest *test* accuracy.

In [ ]:
depths = [1, 2, 3, 4, 6, 10, 20]
train_acc, test_acc = [], []
for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_train, y_train)
    train_acc.append(t.score(X_train, y_train) * 100)
    test_acc.append(t.score(X_test, y_test) * 100)

plt.plot(depths, train_acc, marker="o", color="#F5B41D", label="training penguins")
plt.plot(depths, test_acc, marker="o", color="#102A54", label="hidden penguins")
plt.title("Tree depth vs accuracy")
plt.xlabel("max_depth"); plt.ylabel("Accuracy (%)"); plt.legend(); plt.grid(True)
plt.show()

## Teach 3: Which features matter?

Trees keep score of how useful each feature was. This is one honest answer to "how did the AI decide?"

In [ ]:
best = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)
importance = pd.Series(best.feature_importances_, index=features).sort_values()
plt.barh(importance.index, importance.values, color="#39b54a")
plt.title("Feature importance"); plt.xlabel("Share of decisions")
plt.show()

## SOLUTION: Challenge

**Timer suggestion: 25 minutes.**

### Mild
Train a tree on only **two** features (`flipper_length_mm` and `bill_length_mm`) with `max_depth=2`. Plot it and write the two questions it asks in plain English in a markdown cell.

### Medium
Find the best `max_depth` for the `wize_students.csv` dataset predicting `likes_coding` from `hours_sleep`, `screen_hours`, `focus_score`, and `minutes_reading`. Show your evidence with a chart like Teach 2.

### Spicy A (any platform)
**Unsupervised learning.** Load `shoppers.csv`, which has no labels. Use `KMeans` to find three groups of shoppers, plot them in color, and describe each group in one sentence.

### Spicy B (Google Colab only)
Train a tiny **neural network** on the digits with Keras and compare its accuracy to K nearest neighbors from week 8.

In [ ]:
# MILD: a two feature tree you can read aloud
two = ["flipper_length_mm", "bill_length_mm"]
small = DecisionTreeClassifier(max_depth=2, random_state=42).fit(X_train[two], y_train)
print("Accuracy with two features:", round(small.score(X_test[two], y_test) * 100, 1), "%")
plt.figure(figsize=(10, 4))
plot_tree(small, feature_names=two, class_names=small.classes_, filled=True, fontsize=9)
plt.show()
# Plain English: "Is the flipper shorter than about 206 mm? If yes, is the bill shorter than about 43 mm?
# Short flipper and short bill means Adelie; short flipper and long bill means Chinstrap; long flipper means Gentoo."

In [ ]:
# MEDIUM: best depth for predicting likes_coding
students = load("wize_students.csv")
cols = ["hours_sleep", "screen_hours", "focus_score", "minutes_reading"]
Xs_train, Xs_test, ys_train, ys_test = train_test_split(students[cols], students["likes_coding"], test_size=0.25, random_state=7)
depths = [1, 2, 3, 4, 6, 10]
test_scores = []
for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=7).fit(Xs_train, ys_train)
    test_scores.append(t.score(Xs_test, ys_test) * 100)
plt.plot(depths, test_scores, marker="o", color="#2271b1")
plt.title("likes_coding: depth vs test accuracy"); plt.xlabel("max_depth"); plt.ylabel("Accuracy (%)"); plt.grid(True)
plt.show()
best_depth = depths[test_scores.index(max(test_scores))]
print("Best depth:", best_depth, "with", round(max(test_scores), 1), "%")

In [ ]:
# SPICY A: K means clustering (no labels!)
from sklearn.cluster import KMeans
shoppers = load("shoppers.csv")
kmeans = KMeans(n_clusters=3, random_state=0, n_init=10).fit(shoppers)
shoppers["group"] = kmeans.labels_
plt.scatter(shoppers["age"], shoppers["weekly_spend"], c=shoppers["group"], cmap="viridis")
plt.title("Three shopper groups found by K means"); plt.xlabel("Age"); plt.ylabel("Weekly spend ($)")
plt.show()
print(shoppers.groupby("group")[["age", "weekly_spend"]].mean().round(1))
# Example descriptions: young low spenders, teens who spend a lot, adults in the middle.

In [ ]:
# SPICY B (Google Colab only): a tiny neural network on the digits
# In Colab, TensorFlow and Keras are preinstalled. This cell is skipped in JupyterLite.
try:
    import tensorflow as tf
    from sklearn.datasets import load_digits
    digits = load_digits()
    Xd_train, Xd_test, yd_train, yd_test = train_test_split(digits.data / 16, digits.target, test_size=0.25, random_state=42)

    net = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(64,)),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(10, activation="softmax"),
    ])
    net.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    net.fit(Xd_train, yd_train, epochs=20, verbose=0)
    loss, acc = net.evaluate(Xd_test, yd_test, verbose=0)
    print("Neural network accuracy:", round(acc * 100, 1), "%   (compare with KNN from week 8)")
except ImportError:
    print("TensorFlow is not available here. Open this notebook in Google Colab to run the neural network.")

## Extra activities (if you finish early)

- Set `max_depth=None` (unlimited). Print training and test accuracy. Explain the gap in one sentence.
- In Spicy A, try `n_clusters=2` and `n_clusters=5`. Which number of groups tells the most useful story?
- Which model would you trust more to explain a decision to a parent: the tree or the neural network? Why?

## Reflection

- What is overfitting, in your own words?
- A tree can show its questions. Why might that matter for an AI that decides who gets a loan?